# 10 - Reading and Writing in Microsoft Fabric

This lesson moves from notebook-created examples to files and Delta tables in a Fabric Lakehouse.

## Learning objectives

By the end of this notebook, you will be able to:

- read Lakehouse CSV files with explicit schemas;
- distinguish path-based Delta data from a managed table;
- choose between overwrite and append deliberately; and
- write and read managed Delta tables.

## Prerequisite recap and setup

Earlier notebooks created small DataFrames directly. This lesson reads the supplied files instead.

1. Use a Fabric Spark / PySpark notebook.
2. Attach a default Lakehouse.

## Read a CSV with an implicit schema

In [ ]:
taxi_zones_path = 'abfss://JuniorTrack@onelake.dfs.fabric.microsoft.com/Pyspark_Training.Lakehouse/Files/taxi_zones_raw/taxi_zone_lookup.csv'

t = spark.read.option('header', True).csv(taxi_zones_path)
t.show()
t.printSchema()

## Read a CSV with an explicit schema

An explicit schema documents the source contract and avoids the extra scan and uncertainty of schema inference.

In [ ]:
taxi_schema = '''
    LocationID INT,
    Borough STRING,
    zone STRING,
    service_zone STRING
'''

taxi = (
    spark.read
    .option('header', True)
    .schema(taxi_schema)
    .csv(taxi_zones_path)
)

taxi.show()
taxi.printSchema()
print('taxi rows:', taxi.count())

## Write Delta data to a path

`save` writes Delta files and their transaction log to a path. It does not create a named table in the Lakehouse catalogue.

In [ ]:
taxi_zones_delta_path = 'abfss://JuniorTrack@onelake.dfs.fabric.microsoft.com/Pyspark_Training.Lakehouse/Files/justin/taxi_zones_delta'

(
    taxi.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(taxi_zones_delta_path)
)

taxi_zones_delta = spark.read.format('delta').load(taxi_zones_delta_path)
display(taxi_zones_delta)

## Write a managed Delta table

`saveAsTable('taxi_zones_justin')` writes Delta data and registers a named table in the attached Lakehouse. Downstream code can then use `spark.table('taxi_zones_justin')` without knowing its storage path. In a shared Lakehouse, use your own identifier in the table name.

In [ ]:
(
    taxi.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('taxi_zones_justin')
)

taxi_zones = spark.table('taxi_zones_justin')
taxi_zones.show()
print('Managed-table rows:', taxi_zones.count())

In [ ]:
%%sql
select * from dbo.taxi_zones_justin

## Choose the write mode deliberately

- `overwrite` replaces the target and makes this training notebook safely rerunnable.
- `append` adds rows and can create duplicates when a load is rerun.
- `error` fails if the target exists.
- `ignore` leaves an existing target unchanged.

A typical append uses `new_zones.write.format('delta').mode('append').saveAsTable('taxi_zones_justin')`. Do not run that pattern here unless the rows are genuinely new.

## Your turn - daily pickup-zone performance

The taxi operations team needs a daily view of where trips begin, including trip volume, distance, revenue, and recorded tips. Use the existing `nyc_taxi_trips` Delta table and the lookup table you created from `taxi_zone_lookup.csv`.

Create a managed Delta table named `daily_pickup_zone_performance_<initials>`. Replace `<initials>` with your own identifier so that you do not overwrite another student's table in the shared Lakehouse.

1. Read `nyc_taxi_trips` and your `taxi_zones_<initials>` table with `spark.table()`, then inspect both schemas.
2. Keep trips with a populated `tpep_pickup_datetime` and `PULocationID`, a `trip_distance` greater than zero, and a `total_amount` greater than or equal to zero.
3. Derive `pickup_date` from `tpep_pickup_datetime`.
4. Left join the trips to the zone lookup with `PULocationID = LocationID`. Retain trips whose location ID is absent from the lookup, and label their `Borough` and `zone` as `Unknown`.
5. Produce one row per `pickup_date`, `Borough`, and `zone` with `trip_count`, `average_trip_distance`, `total_revenue`, and `total_recorded_tips`.
6. Sort by latest pickup date, then by total revenue descending. Overwrite your managed Delta table, read it back with `spark.table()`, and display the result.

In [ ]:
# Write your solution here.
# Use a personal suffix in all table names you create in the shared Lakehouse.

### Expected result

Your published table has one row per pickup date and pickup zone, with these columns in this order: `pickup_date`, `Borough`, `zone`, `trip_count`, `average_trip_distance`, `total_revenue`, and `total_recorded_tips`.

Valid trips with a location ID that does not appear in the lookup remain in the result with `Unknown` borough and zone labels. When you read the table back, it has the same row count as the DataFrame you wrote and no duplicate `(pickup_date, Borough, zone)` combinations.

### Solution - reveal after attempting

In [ ]:
from pyspark.sql import functions as F

student_suffix = 'justin'  # Replace with your own initials.
lookup_table = f'taxi_zones_{student_suffix}'
output_table = f'daily_pickup_zone_performance_{student_suffix}'

taxi_trips = spark.table('nyc_taxi_trips')
taxi_zones = spark.table(lookup_table)
taxi_trips.printSchema()
taxi_zones.printSchema()

valid_trips = (
    taxi_trips
    .filter(
        F.col('tpepPickupDateTime').isNotNull()
        & F.col('PULocationID').isNotNull()
        & (F.col('tripDistance') > 0)
        & (F.col('totalAmount') >= 0)
    )
    .withColumn('pickup_date', F.to_date('tpepPickupDateTime'))
)

trips_with_zones = (
    valid_trips.alias('trips')
    .join(
        taxi_zones.alias('zones'),
        F.col('trips.PULocationID') == F.col('zones.LocationID'),
        'left',
    )
    .select(
        F.col('trips.pickup_date'),
        F.coalesce(F.col('zones.Borough'), F.lit('Unknown')).alias('Borough'),
        F.coalesce(F.col('zones.zone'), F.lit('Unknown')).alias('zone'),
        F.col('trips.tripDistance'),
        F.col('trips.totalAmount'),
        F.coalesce(F.col('trips.tipamount'), F.lit(0)).alias('tipamount'),
    )
)

daily_pickup_zone_performance = (
    trips_with_zones
    .groupBy('pickup_date', 'Borough', 'zone')
    .agg(
        F.count('*').alias('trip_count'),
        F.avg('tripDistance').alias('average_trip_distance'),
        F.sum('totalAmount').alias('total_revenue'),
        F.sum('tipamount').alias('total_recorded_tips'),
    )
    .orderBy(F.col('pickup_date').desc(), F.col('total_revenue').desc())
)

(
    daily_pickup_zone_performance.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(output_table)
)

published_performance = spark.table(output_table)
display(published_performance)
print('Rows written:', daily_pickup_zone_performance.count())
print('Rows read back:', published_performance.count())

duplicate_summary_keys = (
    published_performance
    .groupBy('pickup_date', 'Borough', 'zone')
    .count()
    .filter(F.col('count') > 1)
    .count()
)
print('Duplicate summary keys:', duplicate_summary_keys)

## Key takeaway

Use Fabric Lakehouse paths for files, define schemas explicitly, and choose both the Delta target type and write mode deliberately.

You have completed the guided course and are ready for the Delta Lake exercise.